In [ ]:
import pandas as pd
import re,string,nltk,spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.linear_model import LogisticRegression
nltk.download('punkt_tab')
nltk.download('stopwords')

nlp = spacy.load('en_core_web_sm')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")+"/IMDB Dataset.csv"

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [ ]:
df = pd.read_csv(path)

In [ ]:
stop = stopwords.words('english')
def preprocess(text):
  text=text.lower()
  text = re.sub(r'<.,*?>',' ',text)
  text = re.sub(r'http\S+|www\S+'," ",text)
  text = re.sub(r'[^a-z\s]',' ',text)
  tokens = word_tokenize(text)
  tokens = [t for t in tokens if t not in stop]

  doc = nlp(' '.join(tokens))
  lemma = [tok.lemma_ for tok in doc]
  return ' '.join(lemma)

In [ ]:
df['clean_review'] = df['review'].apply(preprocess)
df[['review','clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mention watch oz episode hook rig...
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [ ]:
df['label'] = df['sentiment'].map({'negative':0,'positive':1})
X_train,X_test,y_train,y_test = train_test_split(df['clean_review'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])

tfidf = TfidfVectorizer(max_features=5000)
X_train_t = tfidf.fit_transform(X_train)
X_test_t = tfidf.transform(X_test)
print(X_train_t.shape)

(40000, 5000)


In [ ]:
lr=LogisticRegression(max_iter=1000)
lr.fit(X_train_t,y_train)
pred_lr = lr.predict(X_test_t)
print("LR Accuracy score",accuracy_score(y_test,pred_lr))
print("LR Confusion matrix",confusion_matrix(y_test,pred_lr))
print("LR classification report \n",classification_report(y_test,pred_lr))

LR Accuracy score 0.8888
LR Confusion matrix [[4387  613]
 [ 499 4501]]
LR classification report 
               precision    recall  f1-score   support

           0       0.90      0.88      0.89      5000
           1       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [ ]:
xgb = XGBClassifier(n_estimators=100,max_depth=6,learning_rate=0.1,eval_metric='logloss',random_state=42)
xgb.fit(X_train_t,y_train)
pred_xgb=xgb.predict(X_test_t)
print("XGB Accuracy score",accuracy_score(y_test,pred_xgb))

XGB Accuracy score 0.8279


In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression','XGBoost'],
    'Accuracy':[accuracy_score(y_test,pred_lr),accuracy_score(y_test,pred_xgb)]
    })
print(results)

                 Model  Accuracy
0  Logistic Regression    0.8888
1              XGBoost    0.8279


In [ ]:
def predict_review(review,model):
  clean =preprocess(review)
  vec = tfidf.transform([clean])
  pred = model.predict(vec)[0]
  return 'Positive' if pred==1 else 'Negative'

print(predict_review('This movie was absolutely fantastic!',lr))
print(predict_review('worst movie i have ever watched',xgb))

Positive
Negative
